In [1]:
import sys
import pandas as pd
import pymysql
cnx = pymysql.connect(
    host='192.168.3.139',
    user='sorterbin',
    password='sorterbin123',
    database='machine_status_db',
    port=3306,
    charset='utf8mb4'
 )

query = '''
SELECT A.start_time as 时间, A.alarm_codes AS code, 
       L.alarm_cn AS 内容, A.line_id as line_id, A.machine as machine, L.alarm_type as alarm_type
FROM tester_alarm A JOIN tester_alarm_list L 
ON A.alarm_codes = L.alarm_code
'''

try:    
    cursor = cnx.cursor()
    cursor.execute(query)
    # Fetch the data
    rows = cursor.fetchall()
   
    # Get column names from cursor.description
    columns = [desc[0] for desc in cursor.description]
   
    # Convert to DataFrame with column names
    df_log = pd.DataFrame(rows, columns=columns)
    cursor.close()
    print("✅ Fetched printer summary data from PostgreSQL")
except Exception as e:  
    print(f"Error fetching printer data: {e}")
    sys.exit(1)

print(df_log.head(5))  

✅ Fetched printer summary data from PostgreSQL
                   时间 code                     内容 line_id   machine  \
0 2026-01-09 07:58:26  417                直流模式运行中     1.1  unloader   
1 2026-01-09 07:58:26  417  回料台A1轨道对接位阻挡气缸双信号触发故障     1.1  unloader   
2 2026-01-09 09:24:36  114                     备用     1.2      ivel   
3 2026-01-09 09:24:36  114       回料台A2轨道步进手动参数未设定     1.2      ivel   
4 2026-01-09 09:24:36  114             halm不在空闲状态     1.2      ivel   

      alarm_type  
0   Config/State  
1     Mechanical  
2          Other  
3   Config/State  
4  Vision/Sensor  


In [ ]:
# Extract 3-digit error codes from 内容 like " :,568..." and filter rows without a code
pattern = r'[:：],\s*(\d{3})'

df_log_with_code = df_log.copy()
df_log_with_code['error_code'] = df_log_with_code['内容'].str.extract(pattern, expand=False)

# Keep only rows that contain an error_code
df_log_errors = df_log_with_code.dropna(subset=['error_code']).copy()
df_log_errors['error_code'] = df_log_errors['error_code'].astype(int)

# Keep original columns and rename error_date -> date
df = df_log_errors[['error_date', 'machine_id', 'error_code', '内容', 'occurance']].copy()
df.rename(columns={'error_date': 'date'}, inplace=True)

# Remove embedded error_code from 内容 and trim
df['内容'] = df['内容'].str.replace(pattern, '', regex=True).str.strip()

print(df.head(10))


          date        printer_id  error_code  \
0   2026-03-05  SPT1.2_printer04         568   
1   2026-03-05  SPT1.2_printer04         527   
2   2026-03-05  SPT1.1_printer01         589   
3   2026-03-05  SPT1.1_printer04         527   
4   2026-03-05  SPT1.1_printer03         523   
5   2026-03-05  SPT1.2_printer01         523   
6   2026-03-05  SPT1.1_printer02         572   
8   2026-03-05  SPT1.2_printer04         337   
9   2026-03-05  SPT1.2_printer01         564   
10  2026-03-05  SPT1.1_printer04         283   

                                                   内容  occurance  
0                             Mesh plate not detected        721  
1     Downstream device network communication failure        627  
2                                                            567  
3                                          下游设备网络通讯失败        518  
4                                          屏蔽出板相机数据状态        429  
5   Shielding the camera data status of the outgoi...        378  
6 

In [ ]:
# Split df into top-4 machines for SPT1.1 and SPT1.2 separately

# Filter by line (IMPORTANT: filter using df['machine_id'], not df_log, to avoid index mismatch)
df_11 = df[df['machine_id'].str.startswith('SPT1.1_')].copy()
df_12 = df[df['machine_id'].str.startswith('SPT1.2_')].copy()

# Top 4 machine_ids in each line by TOTAL occurrences (not just number of rows)
top4_ids_11 = df_11.groupby('machine_id')['occurance'].sum().nlargest(4).index.tolist()
top4_ids_12 = df_12.groupby('machine_id')['occurance'].sum().nlargest(4).index.tolist()

# Build dicts for quick access
dfs_by_machine_11 = {mid: df_11.loc[df_11['machine_id'] == mid].copy() for mid in top4_ids_11}
dfs_by_machine_12 = {mid: df_12.loc[df_12['machine_id'] == mid].copy() for mid in top4_ids_12}

# Helper to safely get nth dataframe (pads with empty if missing)
def _get_df(dfs_dict, ids, idx):
    return dfs_dict[ids[idx]] if len(ids) > idx else pd.DataFrame(columns=df.columns)

# Create 4 DataFrames for SPT1.1
df_spt11_machine_1 = _get_df(dfs_by_machine_11, top4_ids_11, 0)
df_spt11_machine_2 = _get_df(dfs_by_machine_11, top4_ids_11, 1)
df_spt11_machine_3 = _get_df(dfs_by_machine_11, top4_ids_11, 2)
df_spt11_machine_4 = _get_df(dfs_by_machine_11, top4_ids_11, 3)

# Create 4 DataFrames for SPT1.2
df_spt12_machine_1 = _get_df(dfs_by_machine_12, top4_ids_12, 0)
df_spt12_machine_2 = _get_df(dfs_by_machine_12, top4_ids_12, 1)
df_spt12_machine_3 = _get_df(dfs_by_machine_12, top4_ids_12, 2)
df_spt12_machine_4 = _get_df(dfs_by_machine_12, top4_ids_12, 3)

print("SPT1.1 top 4 machines:", top4_ids_11)
for i, mid in enumerate(top4_ids_11, 1):
    print(f"df_spt11_machine_{i}: {mid}, shape={dfs_by_machine_11[mid].shape}")

print("SPT1.2 top 4 machines:", top4_ids_12)
for i, mid in enumerate(top4_ids_12, 1):
    print(f"df_spt12_machine_{i}: {mid}, shape={dfs_by_machine_12[mid].shape}")

SPT1.1 top 4 printers: ['SPT1.1_printer04', 'SPT1.1_printer02', 'SPT1.1_printer03', 'SPT1.1_printer01']
df_spt11_printer_1: SPT1.1_printer04, shape=(6685, 5)
df_spt11_printer_2: SPT1.1_printer02, shape=(7315, 5)
df_spt11_printer_3: SPT1.1_printer03, shape=(5672, 5)
df_spt11_printer_4: SPT1.1_printer01, shape=(3625, 5)
SPT1.2 top 4 printers: ['SPT1.2_printer04', 'SPT1.2_printer02', 'SPT1.2_printer01', 'SPT1.2_printer03']
df_spt12_printer_1: SPT1.2_printer04, shape=(7309, 5)
df_spt12_printer_2: SPT1.2_printer02, shape=(7966, 5)
df_spt12_printer_3: SPT1.2_printer01, shape=(5183, 5)
df_spt12_printer_4: SPT1.2_printer03, shape=(5949, 5)


In [7]:
print(df_spt11_printer_4.head(5))

          date        printer_id  error_code               内容  occurance
5   2025-12-25  SPT1.1_printer01         589                         493
10  2025-12-25  SPT1.1_printer01         248  进片位卡尺异常或Mark点异常        138
24  2025-12-25  SPT1.1_printer01         337    出片称重位顶升气缸有异常片         60
27  2025-12-25  SPT1.1_printer01         592           等待上游有片         52
37  2025-12-25  SPT1.1_printer01         283            光幕被触发         35
